## Topic Modeling Using LLMs

In [1]:
import openai
import pandas as pd

from bertopic.representation import OpenAI
from bertopic import BERTopic

from sentence_transformers import SentenceTransformer

In [2]:
df = pd.read_csv('hotel_reviews_with_transl.csv', sep = '\t')
df.sample(3)

,id,hotel,review,lang,reviews_transl,reviews_len
10322,10856,Park Inn,GREAT VALUE FOR MONEY HOTEL The Park Hotel is ...,en,GREAT VALUE FOR MONEY HOTEL The Park Hotel is ...,466
6863,7230,Millemiun,Preis-Leistungsverhältnis optimal...,de,Price-performance ratio optimal...,34
5039,5300,Hilton,Love the Hilton Stayed here between rentals fo...,en,Love the Hilton Stayed here between rentals fo...,501


In [3]:
docs = list(df.reviews_transl)
docs[:3]

["We stayed three nights over Thanksgiving weekend. While it's a ways out of downtown London, the little town it's in is great--lots of delicious restaurants, inexpensive shopping, close to a fairly major highway. Our two rooms were spare but comfortable and very clean. Bathrooms were fine. Mattresses were sad, saggy and lumpy, but at least not rock hard and the sheets fit and stayed on well. The comforter was just the perfect weight for the weather--we all slept well. There's an in-house restaurant with room service, which we never had a chance to use. The a la carte prices were reasonable, except the buffet breakfast, which was high. It's difficult to find the parking lots, but once you figure out the one way streets and round-abouts, there is safe, adequate parking. We'd go back. For only nine pounds a night for two people, it's a great bargain!",
 "very uncomfortable stay We were staying at his hotel only because my father couldnt make it and rather than cancel the rooms we took hi

In [4]:
client = openai.OpenAI(
    # Ollama の OpenAI 互換 API は /v1 パス配下で提供される
    # macOS の Docker は Apple Silicon GPU (Metal) にアクセスできないため、
    # Ollama はホスト側でネイティブ実行し、host.docker.internal 経由で接続する
    base_url = 'http://host.docker.internal:11434/v1',
    api_key = 'ollama',
)

sentence_model = SentenceTransformer("all-MiniLM-L6-v2")
# sentence_model = SentenceTransformer("intfloat/multilingual-e5-large")

# BERTopic の OpenAI RepresentationModel を qwen3 で使う際の注意点:
#
# 1. stop トークンの上書き
#    BERTopic はデフォルトで stop="\n" を設定するため、LLM の出力が最初の改行で
#    打ち切られてしまう。特に qwen3 は thinking mode がデフォルト有効で、
#    レスポンスが "<think>\n" で始まるため即座に停止し空ラベルになる。
#    generator_kwargs で truthy な値 (例: "<|im_end|>") を渡して上書きする。
#    ※ stop=None は falsy なため BERTopic 内部で "\n" に再設定されてしまう。
#
# 2. qwen3 の thinking mode 無効化
#    プロンプト末尾に /no_think を付与して thinking mode を無効化する。
#    これにより <think>...</think> タグが出力されず、ラベルが正しくパースされる。

prompt = """You will extract a short topic label from given documents and keywords.
Here are two examples of topics you created before:

# Example 1
Sample texts from this topic:
- Traditional diets in most cultures were primarily plant-based with a little meat on top, but with the rise of industrial style meat production and factory farming, meat has become a staple food.
- Meat, but especially beef, is the worst food in terms of emissions.
- Eating meat doesn't make you a bad person, not eating meat doesn't make you a good one.

Keywords: meat beef eat eating emissions steak food health processed chicken
topic: Environmental impacts of eating meat

# Example 2
Sample texts from this topic:
- I have ordered the product weeks ago but it still has not arrived!
- The website mentions that it only takes a couple of days to deliver but I still have not received mine.
- I got a message stating that I received the monitor but that is not true!
- It took a month longer to deliver than was advised...

Keywords: deliver weeks product shipping long delivery received arrived arrive week
topic: Shipping and delivery issues

# Your task
Sample texts from this topic:
[DOCUMENTS]

Keywords: [KEYWORDS]

Based on the information above, extract a short topic label (three words at most) in the following format:
topic: <topic_label>
/no_think"""

representation_model = OpenAI(
    client,
    model='qwen3:1.7b',
    prompt=prompt,
    generator_kwargs={"stop": "<|im_end|>"},
)

topic_model = BERTopic(
    embedding_model=sentence_model,
    representation_model=representation_model,
    verbose=True
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
topics, ini_probs = topic_model.fit_transform(docs)

2026-04-08 13:47:04,985 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/383 [00:00<?, ?it/s]

2026-04-08 13:50:26,084 - BERTopic - Embedding - Completed ✓
2026-04-08 13:50:26,085 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-08 13:50:35,368 - BERTopic - Dimensionality - Completed ✓
2026-04-08 13:50:35,370 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-08 13:50:35,636 - BERTopic - Cluster - Completed ✓
2026-04-08 13:50:35,640 - BERTopic - Representation - Fine-tuning topics using representation models.
100%|██████████| 108/108 [56:54<00:00, 31.61s/it]  
2026-04-08 14:47:31,233 - BERTopic - Representation - Completed ✓


In [7]:
topic_model.get_topic_info().head(10).set_index('Topic')[['Count', 'Name', 'Representation']]

,Count,Name,Representation
Topic,,,
-1,6867,-1_Hotel Room Value,[Hotel Room Value]
0,268,0_Hotel Experience,[Hotel Experience]
1,220,1_Hotel Paddington,[Hotel Paddington]
2,187,2_Hotel Reviews,[Hotel Reviews]
3,186,3_Hotel Location and Services,[Hotel Location and Services]
4,180,4_Hotel Location,[Hotel Location]
5,133,5_Hotel Experience,[Hotel Experience]
6,132,6_Noise in hotels,[Noise in hotels]
7,130,7_hilton london value,[hilton london value]
